In [ ]:
RUN_TARGET = "local"  # "colab" | "local"


## Setup Instructions

### Purpose
Use this notebook to generate or refresh probe-row artifacts for all available checkpoints in one place. That means you no longer need to rerun notebooks 04, 05, or 06 just to regenerate IG and attention outputs after training.

### Typical workflow
1. Train or update a checkpoint in notebook 03, 05, or 06.
2. Run this notebook to generate the saved probe-row CSVs in `results/`.
3. Open `08_compare_all_model_probes.ipynb` to visualize the refreshed artifacts.


In [ ]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "pandas": "2.3.2",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "seaborn": "0.13.2",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


# 07 - Centralized Probe Row Generation

This notebook centralizes attribution-score generation for all available checkpoints. It writes row-level probe artifacts once and reuses them downstream.

### What it does
- Generates baseline probe rows if they do not already exist
- Scans `models/` for compatible MTL checkpoints
- Generates MTL probe rows while reusing the already-generated baseline probe rows
- Writes summary CSVs alongside the probe-row CSVs


# 09 - Centralized Probe Row Generation

This notebook centralizes attribution-score generation for all available checkpoints, so you do not need to rerun notebooks 04, 05, and 06 just to refresh IG/attention probe artifacts after training.


In [ ]:
import sys
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break
import importlib
from dataclasses import fields as _dataclass_fields

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)

from xallergen.baseline_notebook_utils import (
    DROPOUT,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    build_tokenizer,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    load_mtl_checkpoint,
    prepare_baseline_probe_frame,
    print_runtime_context,
    run_baseline_probe_suite,
)
from xallergen.mtl_epitope_notebook_utils import (
    MTLOutputPaths,
    run_probe_suite,
    summarize_probe_outputs,
)

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd
import torch


## Runtime Context

This cell resolves the project root, device, data paths, and the split-B epitope-evaluation dataframe shared across all probe runs.


In [ ]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
    print(f"Device: {DEVICE}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for _output_subdir in [
    RESULTS_DIR / "classification",
    RESULTS_DIR / "probing" / "rows",
    RESULTS_DIR / "probing" / "summaries",
    RESULTS_DIR / "figures" / "diagnostics",
]:
    _output_subdir.mkdir(parents=True, exist_ok=True)

POSITIVES_CSV = DATA_DIR / "positives_splitB.csv"
BASELINE_CHECKPOINT_PATH = MODELS_DIR / "baseline_frozen_esm2.pt"
BASELINE_PROBE_ROWS_CSV = RESULTS_DIR / "probing" / "rows" / "baseline_probing_rows.csv"
BASELINE_SUMMARY_CSV = RESULTS_DIR / "probing" / "summaries" / "probing_summary.csv"

tokenizer = build_tokenizer(HF_MODEL_NAME)
epitope_probe_df = prepare_baseline_probe_frame(POSITIVES_CSV)
print(f"Probe proteins in splitB: {len(epitope_probe_df)}")


## Generation Parameters

- `OVERWRITE_EXISTING = False` keeps existing probe artifacts and only generates missing ones.
- `N_RANDOM_DRAWS` controls the random baseline used in the probe summaries.
- `IG_INTERNAL_BATCH_SIZE` controls the memory/speed tradeoff for Captum IG.


In [ ]:
OVERWRITE_EXISTING = True
N_RANDOM_DRAWS = 100
IG_INTERNAL_BATCH_SIZE = 1

VARIANT_LABELS = {
    "baseline": "Baseline (04)",
    "frozen": "MTL (05 frozen)",
    "top1_unfrozen": "MTL (06 top1_unfrozen)",
}


## Checkpoint Mapping

This helper maps checkpoint filenames to their model-family labels and expected output paths in `results/`.


In [ ]:
def infer_variant_from_checkpoint_name(checkpoint_name: str) -> tuple[str | None, str | None]:
    if checkpoint_name == "baseline_frozen_esm2.pt":
        return "baseline", VARIANT_LABELS["baseline"]
    if checkpoint_name == "mtl_frozen_esm2_epitope.pt":
        return "frozen", VARIANT_LABELS["frozen"]
    if checkpoint_name.startswith("mtl_") and checkpoint_name.endswith("_esm2_epitope.pt"):
        variant = checkpoint_name[len("mtl_") : -len("_esm2_epitope.pt")]
        if variant:
            return variant, VARIANT_LABELS.get(variant, f"MTL ({variant})")
    return None, None


def output_paths_for_variant(variant: str, model_family_label: str) -> MTLOutputPaths:
    if variant == "frozen":
        kwargs = {
            "baseline_checkpoint_path": BASELINE_CHECKPOINT_PATH,
            "checkpoint_path": MODELS_DIR / "mtl_frozen_esm2_epitope.pt",
            "metrics_path": RESULTS_DIR / "classification" / "mtl_baseline_metrics.json",
            "probe_rows_path": RESULTS_DIR / "probing" / "rows" / "mtl_probing_rows.csv",
            "baseline_probe_rows_path": RESULTS_DIR / "probing" / "rows" / "baseline_probing_rows.csv",
            "combined_probe_rows_path": RESULTS_DIR / "probing" / "rows" / "mtl_vs_baseline_probing_rows.csv",
            "probe_summary_path": RESULTS_DIR / "probing" / "summaries" / "mtl_probing_summary.csv",
            "compare_summary_path": RESULTS_DIR / "probing" / "summaries" / "mtl_vs_baseline_summary.csv",
            "combined_violins_png": RESULTS_DIR / "figures" / "diagnostics" / "mtl_vs_baseline_probing_violins.png",
            "combined_auroc_density_png": RESULTS_DIR / "figures" / "diagnostics" / "mtl_vs_baseline_probing_auroc_vs_density.png",
            "combined_auprc_density_png": RESULTS_DIR / "figures" / "diagnostics" / "mtl_vs_baseline_probing_auprc_vs_density.png",
            "baseline_summary_csv": BASELINE_SUMMARY_CSV,
            "mtl_family_label": model_family_label,
            "baseline_family_label": VARIANT_LABELS["baseline"],
        }
    else:
        kwargs = {
            "baseline_checkpoint_path": BASELINE_CHECKPOINT_PATH,
            "checkpoint_path": MODELS_DIR / f"mtl_{variant}_esm2_epitope.pt",
            "metrics_path": RESULTS_DIR / "classification" / f"mtl_{variant}_baseline_metrics.json",
            "probe_rows_path": RESULTS_DIR / "probing" / "rows" / f"mtl_{variant}_probing_rows.csv",
            "baseline_probe_rows_path": RESULTS_DIR / "probing" / "rows" / f"baseline_probing_rows_{variant}.csv",
            "combined_probe_rows_path": RESULTS_DIR / "probing" / "rows" / f"mtl_{variant}_vs_baseline_probing_rows.csv",
            "probe_summary_path": RESULTS_DIR / "probing" / "summaries" / f"mtl_{variant}_probing_summary.csv",
            "compare_summary_path": RESULTS_DIR / "probing" / "summaries" / f"mtl_{variant}_vs_baseline_summary.csv",
            "combined_violins_png": RESULTS_DIR / "figures" / "diagnostics" / f"mtl_{variant}_vs_baseline_probing_violins.png",
            "combined_auroc_density_png": RESULTS_DIR / "figures" / "diagnostics" / f"mtl_{variant}_vs_baseline_probing_auroc_vs_density.png",
            "combined_auprc_density_png": RESULTS_DIR / "figures" / "diagnostics" / f"mtl_{variant}_vs_baseline_probing_auprc_vs_density.png",
            "baseline_summary_csv": BASELINE_SUMMARY_CSV,
            "mtl_family_label": model_family_label,
            "baseline_family_label": VARIANT_LABELS["baseline"],
        }
    supported = {field.name for field in _dataclass_fields(MTLOutputPaths)}
    return MTLOutputPaths(**{key: value for key, value in kwargs.items() if key in supported})


## Step 1: Baseline Probe Generation

The baseline probe table is generated once and then reused by MTL probe generation. If the CSV already exists and overwrite is disabled, this step simply loads it from disk.


In [ ]:
if not BASELINE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing baseline checkpoint: {BASELINE_CHECKPOINT_PATH}")

if BASELINE_PROBE_ROWS_CSV.exists() and not OVERWRITE_EXISTING:
    baseline_probe_df = pd.read_csv(BASELINE_PROBE_ROWS_CSV)
    print(f"Reusing existing baseline probe rows: {BASELINE_PROBE_ROWS_CSV}")
else:
    baseline_model, _ = load_baseline_checkpoint(BASELINE_CHECKPOINT_PATH, DEVICE)
    baseline_probe_df = run_baseline_probe_suite(
        model=baseline_model,
        tokenizer=tokenizer,
        eval_df=epitope_probe_df,
        device=DEVICE,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
    )
    baseline_probe_df.to_csv(BASELINE_PROBE_ROWS_CSV, index=False)
    print(f"Saved baseline probe rows to: {BASELINE_PROBE_ROWS_CSV}")

baseline_probe_df.head()


## Step 2: MTL Probe Generation

For each compatible MTL checkpoint in `models/`, this cell generates the model-specific probe rows. The expensive baseline side is reused from Step 1 instead of being recomputed for every variant.


In [ ]:
generation_records = []
for checkpoint_path in sorted(MODELS_DIR.glob("*.pt")):
    variant, model_family = infer_variant_from_checkpoint_name(checkpoint_path.name)
    if variant is None or variant == "baseline":
        continue

    output_paths = output_paths_for_variant(variant, model_family)
    if output_paths.probe_rows_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {variant}: existing probe rows found at {output_paths.probe_rows_path}")
        generation_records.append({
            "variant": variant,
            "checkpoint": str(checkpoint_path),
            "probe_rows_path": str(output_paths.probe_rows_path),
            "status": "reused",
        })
        continue

    model, _ = load_mtl_checkpoint(checkpoint_path, DEVICE, model_name=HF_MODEL_NAME, hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
    probe_outputs = run_probe_suite(
        model=model,
        tokenizer=tokenizer,
        epitope_probe_df=epitope_probe_df,
        baseline_checkpoint_path=BASELINE_CHECKPOINT_PATH,
        device=DEVICE,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        output_paths=output_paths,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
        ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
        resume=False,
        precomputed_baseline_probe_df=baseline_probe_df,
    )
    summarize_probe_outputs(
        probe_df=probe_outputs["probe_df"],
        baseline_probe_df=probe_outputs["baseline_probe_df"],
        output_paths=output_paths,
    )
    generation_records.append({
        "variant": variant,
        "checkpoint": str(checkpoint_path),
        "probe_rows_path": str(output_paths.probe_rows_path),
        "status": "generated",
    })

generation_df = pd.DataFrame(generation_records)
generation_df


## Saved Artifacts

This final cell prints the probe-row CSVs that now exist and can be consumed by `08_compare_all_model_probes.ipynb`.


In [ ]:
print("Saved probe-row artifacts:")
print(f"  {BASELINE_PROBE_ROWS_CSV}")
for row in generation_df.itertuples(index=False):
    print(f"  {row.probe_rows_path} [{row.status}]")
